In [2]:
## run script to create and populate signal_param_rules table for main parquet insertion script
import mysql.connector
from mysql.connector import Error
import time

# Your signal_param_rules dictionary
# Define ALL filtering rules as {signalid: [allowed_eventparams]}
signal_param_rules = {
            # Format: signalid: [allowed_eventparams]
            11: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64],
            19: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64],
            20: [34, 35, 36, 37, 38, 42, 44, 45, 50, 51, 52, 53, 56, 58, 60, 61],
            25: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            35: [34, 35, 36, 37, 38, 40, 42, 43, 45, 47, 48, 50, 52, 53, 54, 56, 58, 61, 62, 64],
            40: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            45: [34, 36, 37, 38, 40, 42, 44, 50, 51, 52, 53, 54, 58, 61, 63, 64],
            47: [34, 36, 37, 38, 42, 48, 49, 52, 53, 54, 56],
            48: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            50: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 64],
            53: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            55: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 63, 64],
            58: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            60: [34, 35, 36, 37, 38, 40, 42, 45, 48, 50, 51, 52, 53, 54, 56, 58, 61, 64],
            80: [42, 45, 46, 50, 51, 56, 60, 61, 62, 64],
            97: [34, 36, 40, 42, 44, 45, 48, 50, 51, 52, 58, 60, 61, 64],
            118: [34, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            123: [34, 40, 44, 45, 46, 58, 60, 61, 62],
            135: [34, 35, 36, 40, 42, 44, 45, 48, 50, 52, 56, 58, 60, 61, 64],
            140: [34, 35, 37, 39, 40, 42, 44, 45, 47, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            150: [34, 36, 37, 40, 42, 44, 45, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            155: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            158: [34, 36, 37, 44, 48, 50, 52, 53, 58, 60],
            160: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            163: [34, 36, 37, 42, 44, 50, 52, 53, 56, 58, 60],
            167: [34, 36, 37, 40, 42, 44, 50, 52, 53, 56, 58, 60],
            185: [34, 36, 40, 42, 44, 45, 50, 52, 56, 58, 60, 61, 64],
            223: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            228: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            236: [36, 42, 44, 45, 50, 52, 56, 58, 60, 61, 64],
            245: [34, 35, 36, 37, 42, 44, 45, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 64],
            250: [34, 35, 37, 38, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            255: [34, 36, 37, 38, 40, 42, 44, 45, 47, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            260: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            265: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 63, 64],
            266: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            267: [34, 37, 38, 42, 43, 48, 53, 54, 56],
            269: [34, 36, 37, 40, 44, 50, 52, 53, 56, 58, 60],
            270: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            275: [34, 36, 37, 38, 42, 43, 44, 45, 46, 48, 50, 52, 53, 54, 56, 58, 60, 61, 64],
            280: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            281: [34, 36, 37, 40, 42, 44, 50, 52, 53, 56, 58, 60],
            282: [36, 37, 42, 48, 53, 54],
            285: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            288: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            290: [34, 35, 36, 37, 38, 40, 42, 43, 44, 47, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            292: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            294: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            295: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            330: [40, 42, 43, 44, 45, 48, 58, 59, 60, 61],
            337: [34, 35, 36, 40, 42, 44, 45, 48, 50, 52, 58, 60, 61, 62, 64],
            340: [42, 44, 45, 46, 50, 51, 56, 60, 61, 62, 64],
            345: [34, 35, 37, 40, 42, 43, 44, 45, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            353: [34, 36, 37, 40, 42, 44, 50, 52, 53, 58, 60, 64],
            354: [34, 36, 37, 42, 44, 50, 52, 53, 58, 60],
            355: [34, 35, 36, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            382: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            396: [42, 44, 45, 46, 50, 56, 60, 61, 62, 64],
            402: [34, 40, 41, 44, 45, 46, 48, 58, 60, 61, 62],
            420: [34, 36, 37, 38, 42, 44, 45, 46, 50, 51, 52, 53, 58, 60, 61],
            425: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 56, 58, 59, 60, 61, 62, 64],
            435: [36, 37, 42, 43, 44, 45, 48, 50, 52, 53, 58, 59, 60, 61, 64],
            440: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            445: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            450: [34, 36, 37, 40, 42, 43, 44, 48, 50, 52, 53, 54, 58, 60, 64],
            455: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            460: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64, 65, 66],
            470: [20, 27, 33, 34, 36, 37, 38, 40, 42, 43, 45, 46, 48, 49, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            475: [34, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 60, 61, 62, 64],
            487: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            492: [34, 35, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 59, 60, 61, 62, 64],
            493: [34, 35, 42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62],
            523: [34, 36, 37, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            535: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 55, 56, 58, 59, 60, 61, 64],
            540: [34, 35, 36, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 64],
            551: [36, 44, 52, 60],
            563: [34, 35, 36, 42, 43, 48, 51, 52, 56],
            565: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            575: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            580: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            581: [34, 35, 36, 37, 38, 40, 42, 44, 48, 50, 52, 53, 54, 58, 60],
            590: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            591: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            594: [42, 44, 45, 46, 50, 56, 57, 60, 61, 62, 64],
            602: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            604: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64, 65, 66],
            608: [36, 42, 44, 45, 52, 58, 60, 61, 64],
            611: [34, 40, 44, 45, 46, 58, 60, 61, 62],
            623: [42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62, 64],
            634: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62],
            639: [34, 36, 40, 42, 43, 44, 45, 46, 50, 52, 56, 58, 60, 61, 62],
            640: [34, 36, 37, 42, 44, 45, 50, 52, 53, 58, 60, 61],
            645: [34, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 54, 56, 58, 59, 60, 64],
            650: [34, 36, 37, 38, 42, 48, 50, 52, 53, 54, 56],
            652: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            655: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64, 65],
            657: [36, 37, 38, 40, 50, 52, 53, 54, 58, 59, 60, 64],
            660: [34, 36, 37, 38, 40, 42, 43, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            661: [34, 35, 36, 37, 38, 42, 43, 44, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            663: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            665: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            668: [34, 36, 37, 38, 40, 42, 43, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            670: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            673: [34, 36, 37, 38, 42, 44, 48, 50, 52, 53, 54, 56, 58, 59, 60, 64],
            675: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            680: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 63, 64],
            715: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            728: [34, 36, 37, 40, 42, 44, 50, 52, 53, 54, 58, 60],
            732: [34, 35, 39, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            785: [44, 50, 56, 60],
            790: [34, 37, 38, 40, 42, 45, 46, 48, 50, 53, 54, 56, 58, 61, 62, 64],
            795: [34, 35, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            800: [42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62, 64],
            868: [36, 37, 38, 40, 50, 52, 53, 54, 58, 59, 64],
            870: [34, 36, 37, 40, 42, 44, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            875: [34, 36, 37, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            880: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            881: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            882: [34, 36, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            885: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            886: [34, 36, 37, 38, 42, 48, 52, 53, 54, 56],
            887: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            892: [34, 36, 37, 38, 40, 42, 44, 45, 50, 52, 53, 54, 56, 58, 60, 64],
            895: [36, 37, 38, 40, 50, 51, 52, 53, 54, 58, 59, 63, 64],
            930: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            935: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            945: [34, 36, 37, 42, 44, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            960: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            962: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            975: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            978: [34, 36, 37, 38, 42, 52, 53, 54],
            983: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 58, 60, 64],
            9999: [34, 35, 36, 37, 38, 42, 44, 45, 48, 50, 51, 52, 53, 54, 58, 60, 61, 64]
        }

def create_database_connection():
    """Create and return a MySQL database connection"""
    try:
        connection = mysql.connector.connect(
            host="localhost",
            user="root",
            password="", ## insert password
            database="" #3 insert database name
        )
        print("✅ Successfully connected to MySQL database")
        return connection
    except Error as e:
        print(f"❌ Error connecting to MySQL: {e}")
        return None

def create_rules_table(connection):
    """Create the signal_param_rules table if it doesn't exist"""
    create_table_query = """
    CREATE TABLE IF NOT EXISTS signal_param_rules (
        id INT AUTO_INCREMENT PRIMARY KEY,
        signalid INT NOT NULL,
        eventparam INT NOT NULL,
        created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
        updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
        UNIQUE KEY unique_rule (signalid, eventparam)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_unicode_ci
    """
    
    try:
        cursor = connection.cursor()
        cursor.execute(create_table_query)
        connection.commit()
        print("✅ Successfully created/verified signal_param_rules table")
        
        # Add indexes if they don't exist
        cursor.execute("SHOW INDEX FROM signal_param_rules WHERE Key_name = 'idx_signalid'")
        if not cursor.fetchone():
            cursor.execute("CREATE INDEX idx_signalid ON signal_param_rules(signalid)")
        
        cursor.execute("SHOW INDEX FROM signal_param_rules WHERE Key_name = 'idx_eventparam'")
        if not cursor.fetchone():
            cursor.execute("CREATE INDEX idx_eventparam ON signal_param_rules(eventparam)")
            
        connection.commit()
        cursor.close()
    except Error as e:
        print(f"❌ Error creating table: {e}")

def populate_rules_table(connection, rules_dict, batch_size=1000):
    """
    Populate the rules table from a Python dictionary
    :param connection: MySQL connection object
    :param rules_dict: Dictionary of {signalid: [eventparams]}
    :param batch_size: Number of records to insert at once
    """
    # Prepare the bulk insert statement with ON DUPLICATE KEY UPDATE
    insert_query = """
    INSERT INTO signal_param_rules (signalid, eventparam)
    VALUES (%s, %s)
    ON DUPLICATE KEY UPDATE 
        updated_at = CURRENT_TIMESTAMP
    """
    
    # Prepare all values as a list of tuples
    values = []
    for signalid, params in rules_dict.items():
        for param in params:
            values.append((signalid, param))
    
    total_rules = len(values)
    print(f"📊 Preparing to insert {total_rules} rules in batches of {batch_size}")
    
    try:
        cursor = connection.cursor()
        start_time = time.time()
        
        # Insert in batches
        for i in range(0, total_rules, batch_size):
            batch = values[i:i + batch_size]
            cursor.executemany(insert_query, batch)
            connection.commit()
            
            # Progress reporting
            percent_complete = min(i + batch_size, total_rules) / total_rules * 100
            print(f"⏳ Inserted batch {i//batch_size + 1}: {min(i + batch_size, total_rules)}/{total_rules} rules ({percent_complete:.1f}%)")
        
        elapsed_time = time.time() - start_time
        print(f"✅ Successfully inserted {total_rules} rules in {elapsed_time:.2f} seconds")
        
        # Verify count
        cursor.execute("SELECT COUNT(*) FROM signal_param_rules")
        count = cursor.fetchone()[0]
        print(f"📊 Total rules in table: {count}")
        
        cursor.close()
    except Error as e:
        print(f"❌ Error inserting rules: {e}")
        connection.rollback()

def verify_rules(connection, sample_signals=None):
    """Verify the inserted rules by checking some sample signals"""
    if sample_signals is None:
        sample_signals = list(signal_param_rules.keys())[:5]  # Check first 5 signals
        
    try:
        cursor = connection.cursor(dictionary=True)
        
        print("\n🔍 Verifying rules for sample signals:")
        for signal in sample_signals:
            cursor.execute("""
                SELECT eventparam 
                FROM signal_param_rules 
                WHERE signalid = %s 
                ORDER BY eventparam
            """, (signal,))
            
            db_params = [row['eventparam'] for row in cursor.fetchall()]
            expected_params = signal_param_rules.get(signal, [])
            
            print(f"\nSignalID {signal}:")
            print(f" - Expected params: {sorted(expected_params)}")
            print(f" - Database params: {sorted(db_params)}")
            
            if sorted(db_params) == sorted(expected_params):
                print("✅ Match")
            else:
                print("❌ Mismatch")
                missing = set(expected_params) - set(db_params)
                extra = set(db_params) - set(expected_params)
                if missing:
                    print(f"   Missing params: {missing}")
                if extra:
                    print(f"   Extra params: {extra}")
        
        cursor.close()
    except Error as e:
        print(f"❌ Error verifying rules: {e}")

def main():
    # Step 1: Create database connection
    connection = create_database_connection()
    if not connection:
        return
    
    try:
        # Step 2: Create the rules table
        create_rules_table(connection)
        
        # Step 3: Populate the table
        populate_rules_table(connection, signal_param_rules)
        
        # Step 4: Verify the data
        verify_rules(connection)
        
    finally:
        if connection.is_connected():
            connection.close()
            print("🔒 MySQL connection closed")

if __name__ == "__main__":
    main()

✅ Successfully connected to MySQL database
✅ Successfully created/verified signal_param_rules table
📊 Preparing to insert 2118 rules in batches of 1000
⏳ Inserted batch 1: 1000/2118 rules (47.2%)
⏳ Inserted batch 2: 2000/2118 rules (94.4%)
⏳ Inserted batch 3: 2118/2118 rules (100.0%)
✅ Successfully inserted 2118 rules in 0.03 seconds
📊 Total rules in table: 2118

🔍 Verifying rules for sample signals:

SignalID 11:
 - Expected params: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64]
 - Database params: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64]
✅ Match

SignalID 19:
 - Expected params: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64]
 - Database params: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64]
✅ Match

SignalID 20:
 - Expected params: [34, 35, 36, 37, 38, 42, 44, 45, 50, 51, 52, 53, 56, 58, 60, 61]
 - Database params: [34, 35, 36, 37, 38, 42, 44, 45, 50, 51, 52, 53, 56, 58, 60, 61]
✅ Match

SignalID 25:
 - Expected params: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 52, 53, 5

In [ ]:
## FOR REFERENCE (not script):

    # Define ALL filtering rules as {signalid: [allowed_eventparams]}
    signal_param_rules = {
            # Format: signalid: [allowed_eventparams]
            11: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64],
            19: [34, 36, 42, 44, 45, 48, 50, 52, 58, 60, 61, 64],
            20: [34, 35, 36, 37, 38, 42, 44, 45, 50, 51, 52, 53, 56, 58, 60, 61],
            25: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            35: [34, 35, 36, 37, 38, 40, 42, 43, 45, 47, 48, 50, 52, 53, 54, 56, 58, 61, 62, 64],
            40: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            45: [34, 36, 37, 38, 40, 42, 44, 50, 51, 52, 53, 54, 58, 61, 63, 64],
            47: [34, 36, 37, 38, 42, 48, 49, 52, 53, 54, 56],
            48: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            50: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 64],
            53: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            55: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 47, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 63, 64],
            58: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            60: [34, 35, 36, 37, 38, 40, 42, 45, 48, 50, 51, 52, 53, 54, 56, 58, 61, 64],
            80: [42, 45, 46, 50, 51, 56, 60, 61, 62, 64],
            97: [34, 36, 40, 42, 44, 45, 48, 50, 51, 52, 58, 60, 61, 64],
            118: [34, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            123: [34, 40, 44, 45, 46, 58, 60, 61, 62],
            135: [34, 35, 36, 40, 42, 44, 45, 48, 50, 52, 56, 58, 60, 61, 64],
            140: [34, 35, 37, 39, 40, 42, 44, 45, 47, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            150: [34, 36, 37, 40, 42, 44, 45, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            155: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            158: [34, 36, 37, 44, 48, 50, 52, 53, 58, 60],
            160: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            163: [34, 36, 37, 42, 44, 50, 52, 53, 56, 58, 60],
            167: [34, 36, 37, 40, 42, 44, 50, 52, 53, 56, 58, 60],
            185: [34, 36, 40, 42, 44, 45, 50, 52, 56, 58, 60, 61, 64],
            223: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            228: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            236: [36, 42, 44, 45, 50, 52, 56, 58, 60, 61, 64],
            245: [34, 35, 36, 37, 42, 44, 45, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 64],
            250: [34, 35, 37, 38, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            255: [34, 36, 37, 38, 40, 42, 44, 45, 47, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            260: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            265: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 63, 64],
            266: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            267: [34, 37, 38, 42, 43, 48, 53, 54, 56],
            269: [34, 36, 37, 40, 44, 50, 52, 53, 56, 58, 60],
            270: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            275: [34, 36, 37, 38, 42, 43, 44, 45, 46, 48, 50, 52, 53, 54, 56, 58, 60, 61, 64],
            280: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            281: [34, 36, 37, 40, 42, 44, 50, 52, 53, 56, 58, 60],
            282: [36, 37, 42, 48, 53, 54],
            285: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            288: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            290: [34, 35, 36, 37, 38, 40, 42, 43, 44, 47, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            292: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            294: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            295: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            330: [40, 42, 43, 44, 45, 48, 58, 59, 60, 61],
            337: [34, 35, 36, 40, 42, 44, 45, 48, 50, 52, 58, 60, 61, 62, 64],
            340: [42, 44, 45, 46, 50, 51, 56, 60, 61, 62, 64],
            345: [34, 35, 37, 40, 42, 43, 44, 45, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            353: [34, 36, 37, 40, 42, 44, 50, 52, 53, 58, 60, 64],
            354: [34, 36, 37, 42, 44, 50, 52, 53, 58, 60],
            355: [34, 35, 36, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            382: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            396: [42, 44, 45, 46, 50, 56, 60, 61, 62, 64],
            402: [34, 40, 41, 44, 45, 46, 48, 58, 60, 61, 62],
            420: [34, 36, 37, 38, 42, 44, 45, 46, 50, 51, 52, 53, 58, 60, 61],
            425: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 56, 58, 59, 60, 61, 62, 64],
            435: [36, 37, 42, 43, 44, 45, 48, 50, 52, 53, 58, 59, 60, 61, 64],
            440: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            445: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            450: [34, 36, 37, 40, 42, 43, 44, 48, 50, 52, 53, 54, 58, 60, 64],
            455: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            460: [34, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 62, 64, 65, 66],
            470: [20, 27, 33, 34, 36, 37, 38, 40, 42, 43, 45, 46, 48, 49, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            475: [34, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 60, 61, 62, 64],
            487: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            492: [34, 35, 37, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 59, 60, 61, 62, 64],
            493: [34, 35, 42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62],
            523: [34, 36, 37, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            535: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 55, 56, 58, 59, 60, 61, 64],
            540: [34, 35, 36, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 59, 60, 61, 64],
            551: [36, 44, 52, 60],
            563: [34, 35, 36, 42, 43, 48, 51, 52, 56],
            565: [34, 35, 36, 37, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 56, 58, 59, 60, 61, 62, 64],
            575: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            580: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            581: [34, 35, 36, 37, 38, 40, 42, 44, 48, 50, 52, 53, 54, 58, 60],
            590: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            591: [34, 36, 40, 42, 44, 45, 46, 48, 50, 51, 52, 56, 58, 60, 61, 62, 64],
            594: [42, 44, 45, 46, 50, 56, 57, 60, 61, 62, 64],
            602: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62, 64],
            604: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64, 65, 66],
            608: [36, 42, 44, 45, 52, 58, 60, 61, 64],
            611: [34, 40, 44, 45, 46, 58, 60, 61, 62],
            623: [42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62, 64],
            634: [34, 36, 42, 44, 45, 46, 48, 50, 52, 58, 60, 61, 62],
            639: [34, 36, 40, 42, 43, 44, 45, 46, 50, 52, 56, 58, 60, 61, 62],
            640: [34, 36, 37, 42, 44, 45, 50, 52, 53, 58, 60, 61],
            645: [34, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 52, 53, 54, 56, 58, 59, 60, 64],
            650: [34, 36, 37, 38, 42, 48, 50, 52, 53, 54, 56],
            652: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            655: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64, 65],
            657: [36, 37, 38, 40, 50, 52, 53, 54, 58, 59, 60, 64],
            660: [34, 36, 37, 38, 40, 42, 43, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            661: [34, 35, 36, 37, 38, 42, 43, 44, 48, 50, 52, 53, 54, 56, 58, 60, 64],
            663: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            665: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            668: [34, 36, 37, 38, 40, 42, 43, 44, 45, 48, 50, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            670: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            673: [34, 36, 37, 38, 42, 44, 48, 50, 52, 53, 54, 56, 58, 59, 60, 64],
            675: [34, 36, 37, 38, 40, 42, 44, 45, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 64],
            680: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 63, 64],
            715: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            728: [34, 36, 37, 40, 42, 44, 50, 52, 53, 54, 58, 60],
            732: [34, 35, 39, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            785: [44, 50, 56, 60],
            790: [34, 37, 38, 40, 42, 45, 46, 48, 50, 53, 54, 56, 58, 61, 62, 64],
            795: [34, 35, 40, 44, 45, 46, 48, 58, 60, 61, 62],
            800: [42, 44, 45, 46, 50, 51, 56, 58, 60, 61, 62, 64],
            868: [36, 37, 38, 40, 50, 52, 53, 54, 58, 59, 64],
            870: [34, 36, 37, 40, 42, 44, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            875: [34, 36, 37, 40, 42, 44, 45, 48, 50, 52, 53, 54, 56, 58, 60, 61, 62, 64],
            880: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            881: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60, 64],
            882: [34, 36, 38, 40, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            885: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            886: [34, 36, 37, 38, 42, 48, 52, 53, 54, 56],
            887: [34, 36, 37, 38, 42, 44, 50, 52, 53, 54, 56, 58, 60],
            892: [34, 36, 37, 38, 40, 42, 44, 45, 50, 52, 53, 54, 56, 58, 60, 64],
            895: [36, 37, 38, 40, 50, 51, 52, 53, 54, 58, 59, 63, 64],
            930: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            935: [34, 36, 37, 40, 42, 44, 45, 46, 48, 50, 52, 53, 56, 58, 60, 61, 62, 64],
            945: [34, 36, 37, 42, 44, 48, 50, 52, 53, 56, 58, 60, 61, 64],
            960: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            962: [34, 36, 40, 42, 44, 45, 46, 48, 50, 52, 56, 58, 60, 61, 62, 64],
            975: [34, 35, 36, 37, 38, 40, 42, 43, 44, 45, 46, 48, 50, 51, 52, 53, 54, 56, 58, 59, 60, 61, 62, 64],
            978: [34, 36, 37, 38, 42, 52, 53, 54],
            983: [34, 36, 37, 38, 40, 42, 44, 50, 52, 53, 54, 58, 60, 64],
            9999: [34, 35, 36, 37, 38, 42, 44, 45, 48, 50, 51, 52, 53, 54, 58, 60, 61, 64]
        }

    
